# Using R and RStudio in Neurodesk

**Author**: Michèle Masson-Trottier

The University of Queensland<br>
<div style="line-height: 2;">
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0
    </a>
</div>

## Purpose

R is widely used in neuroimaging for statistical analysis, data visualisation, and neuroimaging-specific packages. Neurodesk provides R and RStudio through its container system, allowing you to work with neuroimaging data using powerful R packages such as `RNifti`, `oro.nifti`, and `neurobase`.

This tutorial shows how to launch RStudio, install packages to a persistent location, and run basic neuroimaging analyses using R.

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Open RStudio from the Neurodesk application menu
- Install R packages to a persistent library in `~/neurodesktop-storage/`
- Load and inspect NIfTI images using `RNifti` and `oro.nifti`
- Run basic statistical analyses on neuroimaging data
- Use the R Jupyter kernel for reproducible notebook workflows
:::

## Citation and Resources

### Tools and packages used in this workflow

__R Language__
: R Core Team (2024). R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna, Austria. [https://www.r-project.org/](https://www.r-project.org/)

__RNifti__
: Clayden, J.D., et al. (2016). RNifti: Fast R and C++ Access to NIfTI Images. *Journal of Statistical Software*.

__oro.nifti__
: Whitcher, B., et al. (2011). Working with the DICOM and NIfTI Data Standards in R. *Journal of Statistical Software*, 44(6).

### Educational resources

- [R Project](https://www.r-project.org/)
- [RStudio](https://www.rstudio.com/)
- [Neurodesk documentation](https://neurodesk.org)

## Prerequisites

:::{admonition} Before you begin
:class: warning
Make sure you have access to a running Neurodesk instance. See [Getting Set Up with Neurodesk](https://neurodesk.org/getting-started/) for instructions.
:::

- [x] A running Neurodesk environment
- [ ] Basic familiarity with R or RStudio
- [ ] Optional: some neuroimaging data (NIfTI format) to analyse
- [ ] Access to `~/neurodesktop-storage/` directory for persistent package installation

## Opening RStudio in Neurodesk

RStudio is available in Neurodesk through the application menu. There are two ways to launch it:

### Method 1: Application Menu

Navigate to **Applications → Neurodesk → Programming → rstudio** and select the latest version (e.g., **rstudioGUI 4.4.2**).

![Opening RStudio from the Neurodesk application menu.](/static/tutorials/programming/r_rstudio/open_rstudio_menu.png)
*Opening RStudio from the Neurodesk application menu.*

### Method 2: Terminal

Open a terminal in Neurodesktop and run:

```bash
ml rstudio/4.4.2 && rstudio &
```

Replace `4.4.2` with your desired R version (check available versions with `ml avail rstudio`).

![The RStudio IDE running inside Neurodesk with the console, environment panel, and script editor.](/static/tutorials/programming/r_rstudio/rstudio_main_window.png)
*The RStudio IDE running inside Neurodesk with the console, environment panel, and script editor.*

## Setting a Persistent R Library

By default, R packages installed in a Neurodesk container are lost when you restart the container. To save installed packages, configure R to use a persistent library in `~/neurodesktop-storage/`.

### Create the Library Directory

In the RStudio console, run:

```r
# Create a persistent library directory
dir.create("~/neurodesktop-storage/R_libs", showWarnings = FALSE, recursive = TRUE)

# Add it to your R library path
.libPaths(c("~/neurodesktop-storage/R_libs", .libPaths()))

# Verify the library paths
.libPaths()
```

### Make It Permanent

To avoid repeating this every session, create or edit `~/.Rprofile` with the following content:

```r
# Persistent R library in neurodesktop-storage
.libPaths(c("~/neurodesktop-storage/R_libs", .libPaths()))
```

From now on, packages will be automatically installed to the persistent library.

## Installing Neuroimaging Packages

Install the key neuroimaging R packages to your persistent library:

```r
# Install CRAN packages
install.packages(
  c("RNifti", "oro.nifti", "neurobase", "ggplot2", "tidyverse"),
  lib = "~/neurodesktop-storage/R_libs"
)
```

### Installing Bioconductor Packages

Some neuroimaging packages are hosted on Bioconductor. Install BiocManager first:

```r
# Install BiocManager
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager", lib = "~/neurodesktop-storage/R_libs")

# Install Bioconductor packages (example: fMRIdata)
BiocManager::install("fMRIdata", lib = "~/neurodesktop-storage/R_libs")
```

![Installing R packages in the RStudio console.](/static/tutorials/programming/r_rstudio/rstudio_install_packages.png)
*Installing R packages in the RStudio console.*

## Loading and Visualising NIfTI Images

Once you have the neuroimaging packages installed, you can load and visualise NIfTI images directly in R.

### Load a T1-weighted Image

```r
library(RNifti)
library(neurobase)

# Load an anatomical image
img <- readNifti("~/neurodesktop-storage/ds000102/sub-08/anat/sub-08_T1w.nii.gz")

# Check image dimensions and voxel size
dim(img)           # Image dimensions in voxels
pixdim(img)        # Voxel dimensions in mm
```

### Plot an Orthographic View

```r
# Create an orthographic view (sagittal, coronal, axial)
ortho2(img, crosshairs = TRUE)
```

![Orthographic view of a T1w image plotted with the neurobase ortho2() function.](/static/tutorials/programming/r_rstudio/r_ortho_plot.png)
*Orthographic view of a T1w image plotted with the neurobase ortho2() function.*

## Working with Brain Masks

Brain masks (from tools like BET) are commonly used to exclude non-brain voxels from analysis. Here's how to load and apply a mask:

```r
# Load the mask
mask <- readNifti("~/neurodesktop-storage/brain_extraction/bet/T1w_brain_mask.nii.gz")

# Apply the mask to the image
brain <- img * mask

# Visualise the masked brain
ortho2(brain)
```

### Calculate Brain Volume

```r
# Get voxel dimensions in mm
voxel_dim <- pixdim(img)[2:4]  # x, y, z dimensions

# Calculate voxel volume in mm^3
voxel_vol_mm3 <- prod(voxel_dim)

# Calculate voxel volume in mL (1 mL = 1000 mm^3)
voxel_vol_mL <- voxel_vol_mm3 / 1000

# Count brain voxels and calculate total volume
n_brain_voxels <- sum(mask > 0)
brain_vol_mL <- n_brain_voxels * voxel_vol_mL

cat(sprintf("Brain volume: %.1f mL\n", brain_vol_mL))
```

## Statistical Analysis: ROI-based Analysis

A common neuroimaging workflow is to extract signal from regions of interest (ROIs) and perform statistical tests. Here's a simple example:

### Extract Mean Signal in an ROI

```r
# Load fMRI data and ROI mask
fmri <- readNifti("~/neurodesktop-storage/fmri_data/bold.nii.gz")
roi <- readNifti("~/neurodesktop-storage/rois/amygdala_mask.nii.gz")

# Extract voxel values within the ROI
roi_values <- fmri[roi > 0]

# Calculate summary statistics
cat(sprintf(
  "Mean signal in ROI: %.2f ± %.2f\n",
  mean(roi_values, na.rm = TRUE),
  sd(roi_values, na.rm = TRUE)
))
```

### Multiple ROI Analysis

```r
# Load multiple ROI masks
roi_names <- c("amygdala", "hippocampus", "prefrontal_cortex")
results <- data.frame()

for (roi_name in roi_names) {
  roi <- readNifti(paste0("~/neurodesktop-storage/rois/", roi_name, "_mask.nii.gz"))
  roi_values <- fmri[roi > 0]
  results <- rbind(results, data.frame(
    ROI = roi_name,
    Mean = mean(roi_values, na.rm = TRUE),
    SD = sd(roi_values, na.rm = TRUE)
  ))
}

print(results)
```

For more complex statistical analyses (GLM, multiple comparisons, etc.), R offers many packages like `stats`, `lme4`, and `fsl` (R interface to FSL).

## Using the R Kernel in Jupyter Notebooks

Neurodesk also supports R directly in Jupyter notebooks, allowing you to mix R and markdown cells for reproducible reports.

### Starting a Jupyter Notebook with R

Open a terminal and run:

```bash
jupyter notebook
```

Then create a new notebook and select the **R** kernel from the kernel dropdown (if available).

### Checking Available Kernels

To see which kernels are available:

```bash
jupyter kernelspec list
```

If the R kernel is not available, install it via:

```r
install.packages('IRkernel')
IRkernel::installspec()
```

![Selecting the R kernel when creating a new Jupyter notebook in Neurodesk.](/static/tutorials/programming/r_rstudio/jupyter_r_kernel.png)
*Selecting the R kernel when creating a new Jupyter notebook in Neurodesk.*

### Example Jupyter Notebook with R

In a Jupyter notebook with the R kernel, you can mix code and markdown:

```r
# Load neuroimaging packages
library(RNifti)
library(ggplot2)

# Load an image
img <- readNifti("~/neurodesktop-storage/ds000102/sub-08/anat/sub-08_T1w.nii.gz")

# Create a histogram of voxel intensities
hist(img, breaks = 50, main = "Histogram of T1w Intensities")
```

This approach is excellent for creating reproducible analysis reports that combine code, visualisation, and narrative explanation.

## Summary

In this tutorial you:

1. Launched RStudio from within Neurodesk
2. Configured a persistent R library in `~/neurodesktop-storage/` to preserve installed packages
3. Installed neuroimaging R packages (`RNifti`, `oro.nifti`, `neurobase`)
4. Loaded and visualised NIfTI images using R
5. Performed basic statistical analyses (brain volume calculation, ROI extraction)
6. Used the R Jupyter kernel for reproducible notebook workflows

**Key takeaway:** R provides a powerful and flexible environment for neuroimaging analysis, complementing tools like FSL and SPM for custom statistical pipelines and visualisation.

:::{seealso}
- [Python in Jupyter Notebooks](jupyter.ipynb) - for Python-based neuroimaging analysis
- [MATLAB in Neurodesk](matlab.ipynb) - for SPM and other MATLAB-based tools
- [Accessing Neurodesk Tools](accessing_neurodesk_tools.ipynb) - for launching other software environments
:::